# Ayso 3D — صورة ← مجسّم ثلاثي الأبعاد (TripoSR)

## قبل أي شي
**Runtime → Change runtime type → T4 GPU → Save**

بعدها شغّل الخلايا بالترتيب: ▶ الخلية ١، انتظر، ▶ الخلية ٢، انتظر، ▶ الخلية ٣.

**الوقت الكلي: ٤–٦ دقايق** (أول مرة بس، بعدها ٣٠ ثانية).


In [ ]:
# ═══ الخلية ١ — تحضير (٢–٤ دقايق) ═══
import os, sys, subprocess, time
import torch

print("فحص الكارت…")
r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                   capture_output=True, text=True)
print("  ", r.stdout.strip() or "⚠️ ما فيه كارت — رجع لـRuntime وبدّل لـT4")
assert torch.cuda.is_available(), "⚠️ فعّل T4 من Runtime → Change runtime type"

print("\nتثبيت المكتبات… (٣ دقايق)")
subprocess.run([sys.executable,"-m","pip","install","-q",
    "trimesh","rembg","onnxruntime","omegaconf","einops","huggingface_hub","scikit-image"], check=False)
print("  ✓")

print("\nتنزيل TripoSR…")
if not os.path.exists("/content/TripoSR"):
    subprocess.run("cd /content && git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git",
                   shell=True, capture_output=True)
sys.path.insert(0, "/content/TripoSR")
print("  ✓")

# ── إصلاح torchmcubes (يحتاج ترجمة CUDA ويفشل بكولاب) ──
print("\nبديل torchmcubes…")
SHIM = r"""
import torch, numpy as np
from skimage import measure
def marching_cubes(vol, thresh, **kw):
    v = np.asarray(vol.detach().cpu().numpy() if hasattr(vol, "detach") else vol)
    if v.ndim == 4: v = v[0]
    v = v.astype(np.float32)
    if not np.isfinite(v).all() or v.min() >= thresh or v.max() <= thresh:
        return torch.zeros((0,3), dtype=torch.float32), torch.zeros((0,3), dtype=torch.int64)
    verts, faces, _, _ = measure.marching_cubes(v, level=float(thresh))
    return torch.from_numpy(verts.astype(np.float32)), torch.from_numpy(faces.astype(np.int64))
"""
open("/content/torchmcubes.py","w").write(SHIM)
if "/content" not in sys.path: sys.path.insert(0,"/content")
import importlib, skimage; importlib.invalidate_caches()
print("  ✓")

print("\nتحميل النموذج (١.٧ غ)…")
from tsr.system import TSR
MODEL = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
MODEL.renderer.set_chunk_size(32768)
MODEL.to("cuda"); MODEL.eval()
print("  ✓ جاهز على T4")
print("\n✅ الخلية ١ خلصت — روح للخلية ٢")

In [ ]:
# لو فشلت الخلية ١ بـ ModuleNotFoundError: torchmcubes — شغّل هذي بس
# ═══ إصلاح: torchmcubes ← skimage (يشتغل بلا ترجمة CUDA) ═══
import sys, subprocess, os
subprocess.run([sys.executable,"-m","pip","install","-q","scikit-image"], check=False)

open("/content/torchmcubes.py","w").write('import torch, numpy as np\nfrom skimage import measure\n\ndef marching_cubes(vol, thresh, **kw):\n    v = np.asarray(vol.detach().cpu().numpy() if hasattr(vol, "detach") else vol)\n    if v.ndim == 4:\n        v = v[0]\n    v = v.astype(np.float32)\n    if not np.isfinite(v).all() or v.min() >= thresh or v.max() <= thresh:\n        return torch.zeros((0, 3), dtype=torch.float32), torch.zeros((0, 3), dtype=torch.int64)\n    verts, faces, _, _ = measure.marching_cubes(v, level=float(thresh))\n    return (torch.from_numpy(verts.astype(np.float32)),\n            torch.from_numpy(faces.astype(np.int64)))\n')
if "/content" not in sys.path: sys.path.insert(0, "/content")
import importlib; importlib.invalidate_caches()
import torchmcubes
print("torchmcubes بديل ✓ |", torchmcubes.__file__)

# ── تحميل النموذج ──
import torch
from tsr.system import TSR
MODEL = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
MODEL.renderer.set_chunk_size(32768)
MODEL.to("cuda"); MODEL.eval()
print("✓ جاهز على T4 — روح للخلية ٢")


In [ ]:
# ═══ الخلية ٢ — ارفع صورتك ═══
from google.colab import files
up = files.upload()
IMAGE = "/content/" + list(up.keys())[0]
print("\nالصورة:", IMAGE)

from PIL import Image
im = Image.open(IMAGE)
print("الحجم:", im.size)
display(im.convert("RGB").resize((320, int(320*im.size[1]/im.size[0]))))
print("\n✅ روح للخلية ٣")

In [ ]:
# ═══ الخلية ٣ — التوليد (٣٠ ثانية) ═══
import numpy as np, trimesh, time
from PIL import Image
from tsr.utils import remove_background, resize_foreground

img = Image.open(IMAGE).convert("RGBA")
# إزالة الخلفية لو الصورة RGBA فاضية
arr0 = np.array(img)
if arr0[:,:,3].max() == 0:
    try:
        import rembg
        img = rembg.remove(img, session=rembg.new_session())
        print("خلفية انشالت ✓")
    except Exception as e:
        print("rembg فشل:", str(e)[:60])

proc = remove_background(img)
proc = resize_foreground(proc, 0.85)
a = np.array(proc.convert("RGB")).astype(np.float32)/255.0
a = a * np.array(proc.convert("RGBA"))[:,:,3:4].astype(np.float32)/255.0

print("التوليد…")
t0 = time.time()
with torch.no_grad():
    codes = MODEL([torch.from_numpy(a[np.newaxis]).cuda()], device="cuda")
    mesh = MODEL.extract_mesh(codes, has_vertex_color=True, resolution=256)[0]
print(f"خلص بـ {time.time()-t0:.1f} ثانية")
print(f"رؤوس {len(mesh.vertices):,} · وجوه {len(mesh.faces):,}")

mesh.export("/content/model.glb")
mesh.export("/content/model.obj")
print(f"\n✅ model.glb · {os.path.getsize('/content/model.glb')/1e6:.1f} MB")

# عرض ٣ زوايا
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
V = np.asarray(mesh.vertices); F = np.asarray(mesh.faces)
if len(F) > 5000: F = F[np.random.choice(len(F), 5000, replace=False)]
fig = plt.figure(figsize=(15,5), facecolor="white")
for i,(el,az,name) in enumerate([(18,35,"أمامي"),(0,0,"جانبي"),(75,20,"علوي")]):
    ax = fig.add_subplot(1,3,i+1, projection="3d")
    pc = Poly3DCollection(V[F], alpha=1.0)
    pc.set_facecolor("#4a90d9"); pc.set_edgecolor("none")
    ax.add_collection3d(pc)
    ax.auto_scale_xyz(V[:,0],V[:,1],V[:,2])
    ax.set_box_aspect((1,1,1)); ax.view_init(el,az); ax.set_axis_off()
    ax.set_title(name, fontsize=14)
plt.tight_layout()
plt.savefig("/content/samples.png", dpi=100, facecolor="white", bbox_inches="tight")
plt.show()

print("\n✅ روح للخلية ٤")

In [ ]:
# ═══ الخلية ٤ — نزّل الملفات ═══
import os, subprocess
subprocess.run("cd /content && zip -q -j ayso-3d.zip model.glb model.obj samples.png", shell=True)
print("الحجم:", round(os.path.getsize("/content/ayso-3d.zip")/1e6,1), "MB")
from google.colab import files
files.download("/content/ayso-3d.zip")
files.download("/content/model.glb")